---
abstract: ''
keywords: ''
authors:
  - name: Marco Betschart
exports:
- format: pdf
  template: arxiv_two_column
  output: Advanced-Deep-Learning-Summary.pdf
---

# Summary - Advanced Deep Learning

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchinfo import summary
from torcheval.metrics import MulticlassAccuracy

import numpy as np

import wandb

# Network Architecture Design Patterns

## Output Size Decrease

**Problem:** The input image (e.g., 1000x1000) is too large to feed directly into a classifier (MLP), which would result in millions of parameters,.

**Why:** High resolution contains redundant information; we need to condense features.

**Solution:** Use **Pooling** (Max/Average) or **Strided Convolutions**,. Strided convolutions are preferred in modern networks as they allow the network to learn how to downsample adaptively.

In [3]:
# Option A: Max Pooling
torch.nn.MaxPool2d(kernel_size=2, stride=2)
# Option B: Strided Convolution (Preferred)
torch.nn.Conv2d(in_channels=64, out_channels=128,
  kernel_size=3, stride=2, padding=1)

Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

## Normalization

**Problem:** Deep networks are difficult to train and may not converge.

**Why:** Data distribution shifts as it travels through layers (internal covariate shift), and inputs to later layers are not normalized.

**Solution:** **Batch Normalization**. It learns to normalize data (mean 0, std 1) inside the network, enabling higher learning rates and faster training.

In [4]:
torch.nn.Sequential(
  torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
  torch.nn.BatchNorm2d(64), # Batch Normalization
  torch.nn.ReLU())

Sequential(
  (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
)

## Residual Connections

**Problem:** Adding more layers to a deep network (e.g., >30) can degrade performance and stop convergence,.

**Why:** Gradients vanish during backpropagation, and it is difficult for layers to learn the identity function (doing nothing) if needed

**Solution:** Add the input $x$ to the output of the layer $F(x)$, creating a **Skip Connection** ($y = F(x) + x$). This allows gradients to flow through the network easily.

In [5]:
class ResidualBlock(torch.nn.Module):
  def forward(self, x):
    identity = x
    out = self.conv_layers(x) # F(x)
    return out + identity     # F(x) + x

## Output Size Increase

**Problem:** Generative models (like autoencoders) need to reconstruct an image from a small latent vector.

**Solution:** **Upsampling** (repetition) or **Transposed Convolutions**. Transposed convolutions learn weights to optimally upsample the data.

In [6]:
# Option A: Upsampling
torch.nn.Upsample(scale_factor=2, mode='nearest')
# Option B: Transposed Convolution (Learnable)
torch.nn.ConvTranspose2d(in_channels=64,
  out_channels=32, kernel_size=3, stride=2)

ConvTranspose2d(64, 32, kernel_size=(3, 3), stride=(2, 2))

## Channel Number Decrease (Bottleneck)

**Problem:** Computational cost is too high when convolutional filters operate on tensors with large depth (e.g., 256 channels).

**Solution:** **Bottleneck Layers**. Use a $1 \times 1$ convolution to reduce the number of channels (e.g., to 64), perform the expensive $3 \times 3$ convolution, and then scale back up.

In [7]:
bottleneck = torch.nn.Sequential(
  # Compress:
  torch.nn.Conv2d(256, 64, kernel_size=1),
  # Process:
  torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
  # Expand:
  torch.nn.Conv2d(64, 256, kernel_size=1))

## Deep Networks with Fewer Parameters

**Problem:** Large filters (e.g., $11 \times 11$) have a massive number of parameters ($11 \times 11 = 121$ weights per channel).

**Solution:** Stack multiple small filters (e.g., $3 \times 3$). Two stacked $3 \times 3$ layers have a receptive field of $5 \times 5$ but fewer parameters and more non-linearities (activation functions), making training easier.

In [8]:
# Replaces one large 5x5 convolution
stack = torch.nn.Sequential(
  torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
  torch.nn.ReLU(),
  torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
  torch.nn.ReLU())

## Multiple Resolution (Inception)

**Problem:** It is unclear which kernel size ($1 \times 1$, $3 \times 3$, or $5 \times 5$) is best for a specific feature.

**Solution:** **Inception Modules**. Compute convolutions with different kernel sizes in parallel and concatenate the results.

In [9]:
class InceptionModule(torch.nn.Module):
  def forward(self, x):
    p1 = self.conv1x1(x)
    p2 = self.conv3x3(x)
    p3 = self.conv5x5(x)
    # Concatenate along channel dimension:
    return torch.cat([p1, p2, p3], dim=1)

## Faster Large Convolutions

**Problem:** Standard 2D convolutions are computationally expensive.

**Solution:** **Separable Convolutions**.
**Spatial Separation:** Replace an $N \times N$ filter with a $1 \times N$ and $N \times 1$ filter.
**Depth-wise Separation:** Use a single filter per channel (Depth-wise) followed by a $1 \times 1$ filter to mix channels (Point-wise),.

In [10]:
separable = torch.nn.Sequential(
  # Depthwise: groups=in_channels => 1 filter per channel
  torch.nn.Conv2d(32, 32, kernel_size=3, groups=32),
  # Pointwise: 1x1 conv to mix channels
  torch.nn.Conv2d(32, 64, kernel_size=1))

## Share Features (Multi-Head)

**Problem:** You need multiple outputs (e.g., move selection AND win probability in chess) from the same input.

**Solution:** **Multi-headed Networks**. Use a shared "backbone" to extract features, then split into separate "heads" (MLPs) for different tasks. Gradients from both heads help train the backbone.

In [11]:
class MultiHeadNet(torch.nn.Module):
  def forward(self, x):
    features = self.shared_backbone(x)
    class_out = self.classification_head(features)
    reg_out = self.regression_head(features)
    return class_out, reg_out

## Generate Sparse Layers (Dropout)

**Problem:** Overfitting; the network relies too heavily on specific neurons.

**Solution:** **Dropout**. Randomly set the output of some neurons to zero during training. This forces the network to generalize and become robust to missing data.

In [12]:
torch.nn.Dropout(p=0.5)

Dropout(p=0.5, inplace=False)

## Constrain Model Parameters

**Problem:** Overfitting due to large weights.

**Solution:** **L2 Regularization (Weight Decay)**. Add a penalty term to the loss function proportional to the size of the weights, forcing them towards zero.

In [13]:
model = torch.nn.TransformerEncoder(torch.nn.TransformerEncoderLayer(d_model=512, nhead=8), num_layers=6)

/Users/marbetschar/Development/marbetschar/notes/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [14]:
optimizer = torch.optim.SGD(
   model.parameters(), lr=0.01, weight_decay=1e-5)

# PyTorch Training Pipeline

In [15]:
# Make training reproducible:
seed = 1234
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# Preprocess data:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor()])
data_train = torchvision.datasets.MNIST(
    root='data/mnist', download=True,
    transform=transform)
# data_test = ...MNIST(...train=False)

In [16]:
data_test = torchvision.datasets.MNIST(root='data/mnist', train=False, download=True, transform=transform)

class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Conv2d(1, out_channels=4, kernel_size=3),
      nn.ReLU(),
      nn.MaxPool2d(2, 2),
      nn.Conv2d(4, 8, 3),
      nn.ReLU(),
      nn.MaxPool2d(2, 2),
      nn.Conv2d(8, 8, 3),
      nn.ReLU(),
      nn.Flatten(),
      nn.Linear(72, 120),
      nn.ReLU(),
      nn.Linear(120, 10)
    )

  def forward(self, x):
    return self.layers.forward(x)
model = SimpleCNN()

In [17]:
# Split data into train and validation sets:
len_train = (int)(0.8 * len(data_train))
len_val = len(data_train) - len_train
data_train_subset, data_val_subset =(
  torch.utils.data.random_split(
    data_train, [len_train, len_val]))

# Construct data loaders for data sets:
BATCH_SIZE = 64
data_train_loader = torch.utils.data.DataLoader(
  dataset=data_train_subset, shuffle=True,
  batch_size=BATCH_SIZE)
data_val_loader = torch.utils.data.DataLoader(
  dataset=data_val_subset, shuffle=False,
  batch_size=BATCH_SIZE)
data_test_loader = torch.utils.data.DataLoader(
  data_test, batch_size=64)

wandb.login()
def train(epochs: int, model, loss_fn, optim,
    metrics, device):
  wandb.init(project="mnist-example",
    config={'epochs': epochs,
        'batch_size':
          data_train_loader.batch_size})
  step_count = 0
  model = model.to(device)
  # Training:
  for epoch in range(epochs):
      model.train()
      metrics.reset()
      for step, (inputs, labels) in\
              enumerate(data_train_loader):
          inputs = inputs.to(device)
          labels = labels.to(device)

          # Zero your gradients for every batch!
          optim.zero_grad()

          outputs = model(inputs)
          _, predicted = torch.max(outputs, 1)

          train_loss = loss_fn(outputs, labels)
          train_loss.backward()
          optim.step()

          metrics.update(predicted, labels)
          train_acc = metrics.compute()

          train_metrics = {
              'train/train_loss:': train_loss,
              'train/train_acc': train_acc,
              'train/epoch': epoch}

          step_count += 1
          wandb.log(train_metrics, step=step_count)
      # Validation:
      model.eval()
      metrics.reset()
      val_loss = []
      val_steps = 0
      for step, (inputs, labels) in\
              enumerate(data_val_loader):
          inputs = inputs.to(device)
          labels = labels.to(device)
          with torch.no_grad():
              outputs = model(inputs)
              _, predicted = torch.max(outputs, 1)

              val_loss.append(
                loss_fn(outputs, labels).item())
              metrics.update(predicted, labels)
          val_steps += 1

      val_acc = metrics.compute()
      val_loss_mean = np.mean(val_loss)
      val_metrics = {'val/val_loss': val_loss_mean,
                     'val/val_acc' : val_acc}
      wandb.log(val_metrics, step=val_steps)
      print(f"Epoch {epoch:02} ...")
  wandb.finish()

wandb: Currently logged in as: marbetschar (marbetschar-zhaw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [18]:
model = SimpleCNN()
metrics = MulticlassAccuracy(num_classes=10)
optim = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
device = 'cuda' if torch.cuda.is_available() else\
    'mps' if torch.mps.is_available() else 'cpu'
epochs = 10

train(epochs, model, loss_fn, optim, metrics, device)

Epoch 00 ...
Epoch 01 ...


wandb: WARNING Tried to log to step 188 that is less than the current step 750. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 188 that is less than the current step 1500. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


Epoch 02 ...
Epoch 03 ...
Epoch 04 ...


wandb: WARNING Tried to log to step 188 that is less than the current step 2250. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 188 that is less than the current step 3000. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 188 that is less than the current step 3750. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


Epoch 05 ...
Epoch 06 ...
Epoch 07 ...


wandb: WARNING Tried to log to step 188 that is less than the current step 4500. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 188 that is less than the current step 5250. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 188 that is less than the current step 6000. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


Epoch 08 ...
Epoch 09 ...


train/epoch,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇█████
train/train_acc,▁▂▃▃▃▇▇▇▇▇▇▇▇▇▇▇████████████████████████
train/train_loss:,██▄▃▂▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁
train/epoch,9
train/train_acc,0.98475
train/train_loss:,0.02843


# MLP: Multi Layer Perceptron

- ‘Simple’ Problems where the input is features
- Output layers in a CNN after feature extractions
- Feature transformation (for example after attention layers)
- Dimensionality reduction

Each node calculates its output $y$ based in the inputs $x$, the weights $w$ (on the edges), a bias value $b$ and the activation function $\sigma$:

$$
y = \sigma \left( \sum_{k=1}^{n} w_k x_k + b \right)
$$

:::{prf:definition} MLP: Nr of Params
$$
\text{Nr of Params} = (N_{in} \times N_{out}) + N_{out}
$$

$N_{in}$ is Input Size, $N_{out}$ is Output Size (the addition at the end is because there is one bias per output).

**Example:** Flatten a $10 \times 10$ image (100 pixels) and feed it into a Dense layer with $50$ neurons = $5,050 \text{ parameters}$.
:::

:::{prf:theorem} Universal Approximation Theorem
There exists an activation function $\sigma$ which is analytic, strictly increasing and sigmoidal and has the following property: For any $f \in C[0,1]^d$ and $\varepsilon > 0$ there exist constants $d_i, c_{ij}, \theta_{ij}, \gamma_i$ and vectors $\mathbf{w}^{ij} \in \mathbb{R}^d$ for which

$$
\left| f(\mathbf{x}) - \sum_{i=1}^{6d+3} d_i \, \sigma \left( \sum_{j=1}^{3d} c_{ij} \sigma( \mathbf{w}^{ij} \cdot \mathbf{x} - \theta_{ij}) - \gamma_i \right) \right| < \varepsilon
$$

for all $\mathbf{x} = (x_1, \ldots, x_d) \in [0,1]^d$.

A simplified representation is

$$
f(\mathbf{x}) \approx \sum_{i=1}^{N(\varepsilon)} a_i \, \sigma(\mathbf{w}_i \cdot \mathbf{x} + b_i).
$$
:::

# CNN: Convolutional Neural Network

In [19]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Conv2d(1,out_channels=4,kernel_size=3),
      nn.ReLU(),
      nn.MaxPool2d(2, 2), nn.Conv2d(4, 8, 3),
      nn.ReLU(),
      nn.MaxPool2d(2, 2), nn.Conv2d(8, 8, 3),
      nn.ReLU(), nn.Flatten(),
      nn.Linear(72, 120),nn.ReLU(),nn.Linear(120, 10))
  def forward(self, x): return self.layers.forward(x)
summary(SimpleCNN(), input_size=(64, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleCNN                                [64, 10]                  --
├─Sequential: 1-1                        --                        --
│    └─Conv2d: 2-1                       [64, 4, 26, 26]           40
│    └─ReLU: 2-2                         [64, 4, 26, 26]           --
│    └─MaxPool2d: 2-3                    [64, 4, 13, 13]           --
│    └─Conv2d: 2-4                       [64, 8, 11, 11]           296
│    └─ReLU: 2-5                         [64, 8, 11, 11]           --
│    └─MaxPool2d: 2-6                    [64, 8, 5, 5]             --
│    └─Conv2d: 2-7                       [64, 8, 3, 3]             584
│    └─ReLU: 2-8                         [64, 8, 3, 3]             --
│    └─Flatten: 2-9                      [64, 72]                  --
│    └─Linear: 2-10                      [64, 120]                 8,760
│    └─ReLU: 2-11                        [64, 120]                 --
│    └─Lin

## Why CNNs?

In Deep Learning you usually start with raw data and you aim to learn features first - then based on these learned features, you would then try to solve the actual problem for which we can again use MLPs for example (i.e. in classification).

## Fundamental Design Principles

**Data Transformation:** Standard CNNs typically decrease spatial resolution (width/height) while increasing the number of channels (depth) to extract higher-level features.

**Downsampling:** Accomplished via **Pooling** (Max/Average) or **Strided Convolutions**. Strided convolutions (e.g., $stride=2$) are preferred in modern networks as the weights are learnable, allowing the network to adaptively reduce resolution.

**Normalization:** **Batch Normalization** is critical for training deep networks. it ensures data remains normalized (mean 0, variance 1) as it travels through layers, accelerating convergence and enabling higher learning rates.

## Down-/Upsampling

**Max Pooling (Downsampling)**

$$
\begin{array}{ccc}
\begin{bmatrix}
6 & 2 & 3 & 2 \\
1 & 4 & 5 & 1 \\
1 & 2 & 3 & 4 \\
1 & 0 & 5 & 6
\end{bmatrix}
&
\xrightarrow{\text{Max Pooling (2$\times$2)}}
&
\begin{bmatrix}
6 & 5 \\
2 & 6
\end{bmatrix}
\end{array}
$$

**Max Unpooling (Upsampling)**

Max pooling remembers max position. Max unpooling places value at correct position. Zeros fill non-max positions.

$$
\begin{array}{cccc}
\xrightarrow{\text{Max Pool (2$\times$2)}}
&
\begin{bmatrix}
6 & 5 \\
2 & 6
\end{bmatrix}
&
\xrightarrow{\text{Max Unpool (2$\times$2)}}
&
\begin{bmatrix}
6 & 0 & 0 & 2 \\
0 & 4 & 0 & 0 \\
0 & 0 & 3 & 0 \\
0 & 0 & 0 & 6
\end{bmatrix}
\end{array}
$$

**Average Pooling (Downsampling)**

$$
\begin{array}{ccc}
\begin{bmatrix}
6 & 2 & 3 & 2 \\
1 & 4 & 5 & 1 \\
1 & 2 & 3 & 4 \\
1 & 0 & 5 & 6
\end{bmatrix}
&
\xrightarrow{\text{Avg Pooling (2$\times$2)}}
&
\begin{bmatrix}
3 & 3 \\
1 & 5
\end{bmatrix}
\end{array}
$$

**Nearest Neighbor (Upsampling)**

Replicates each value into 2x2 block ($\times 2$ scale).

$$
\begin{array}{ccc}
\begin{bmatrix}
6 & 5 \\
2 & 6
\end{bmatrix}
&
\xrightarrow{\text{Nearest Neighbour}}
&
\begin{bmatrix}
6 & 6 & 5 & 5 \\
6 & 6 & 5 & 5 \\
2 & 2 & 6 & 6 \\
2 & 2 & 6 & 6
\end{bmatrix}
\end{array}
$$

**Bed of Nails (Upsampling)**

Zeroes fill unknown positions.

$$
\begin{array}{ccc}
\begin{bmatrix}
6 & 5 \\
2 & 6
\end{bmatrix}
&
\xrightarrow{\text{Bed of Nails}}
&
\begin{bmatrix}
6 & 0 & 5 & 0 \\
0 & 0 & 0 & 0 \\
2 & 0 & 6 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}
\end{array}
$$

**Transposed Convolution (Upsampling)**

A transposed convolution performs the **reverse operation** of a standard convolution: it maps each input element to multiple output positions. For each input value, the kernel is multiplied and placed at different output locations.

Transposed convolutions can be understood as matrix multiplication. A standard convolution can be represented as: $\mathbf{y} = \mathbf{C} \mathbf{x}$ where $\mathbf{C}$ is the convolution matrix (Toeplitz matrix). The transposed convolution is: $\mathbf{y} = \mathbf{C}^T \mathbf{x}$. The transpose relationship gives these operations their name, though they are **not true inverses** - they only approximate inversion through backpropagation.

_Example: 2x2 Input, 3x3 Kernel, Stride 1_

$$
\begin{array}{cc}
\begin{bmatrix}
a & b \\
c & d \\
\end{bmatrix}
&
\begin{bmatrix}
w1 & w2 & w3 \\
w4 & w5 & w6 \\
w7 & w8 & w9 \\
\end{bmatrix}
\end{array}
$$

**Process:** Each input value is multiplied by the entire kernel and placed at its output position:
$a$ is multiplied by the full kernel and placed starting at output position $(0,0)$, $b$ is multiplied and placed at $(0,2)$ (shifted by stride), $c$ is multiplied and placed at $(2,0)$, $d$ is multiplied and placed at $(2,2)$

Overlapping regions are summed, producing a $4 \times 4$ output:

$$
\begin{bmatrix}
aw_1 & aw_2+bw_1 & aw_3+bw_2 & bw_3 \\
aw_4+cw_1 & aw_5+bw_1+cw_4+dw_1 & \cdots & \cdots \\
aw_7+cw_4 & \cdots & \cdots & \cdots \\
cw_7 & cw_8+dw_7 & cw_9+dw_8 & dw_9
\end{bmatrix}
$$

## Training Stability: Residual Connections

**Problem:** Vanishing/exploding gradients make training very deep networks (e.g., >30 layers) difficult; adding layers can actually degrade performance.

**Solution:** **Residual (Skip) Connections** add the original input $x$ to the output of a layer block $F(x)$, resulting in $y = F(x) + x$. This allows gradients to flow more easily through "shortcuts".

## CNN Formulae

:::{prf:definition} Standard Convolution: Output Size

For an $N \times N$ image, $K \times K$ filter, padding $P$ and stride $S$, the convolution output size is:

$$
\left\lfloor \frac{N + 2P - K}{S} + 1 \right\rfloor
\times
\left\lfloor \frac{N + 2P - K}{S} + 1 \right\rfloor
$$

For input $32 \times 32$, $K=3, S=2, P=1 \implies$ $\text{Output Size} = \lfloor \frac{32 + 2 \cdot 1 - 3}{2} \rfloor + 1 = 16$
:::

:::{prf:definition} Transposed Convolution: Output Size
Calculate the output **Height ($H$)** and **Width ($W$)** using the same formula for each dimension:

$$N_{out} = (N_{in} - 1) \times S - 2 \times P + K + P_{out}$$

$P_{out}$ is a parameter used to resolve **shape ambiguity** that occurs during the upsampling process: When you perform a standard strided convolution, the output size is calculated using a "floor" operation (rounding down). Because of this rounding, multiple different input sizes can actually result in the exact same output size.

**Example:** **Input:** $H \times W = 10 \times 20$, **Kernel:** $3 \times 3$, **Stride:** $2 \times 2$ (Standard square kernel/stride)

$H_{out} = (10 - 1) \times 2 + 3 = 21 \quad W_{out} = (20 - 1) \times 2 + 3 = 41$
:::

:::{prf:definition} Standard/Transposed Convolution: Nr of Params

$$
\text{Nr of Params} = (K \times K \times C_{in} + 1_{bias}) \times C_{out}
$$

For `nn.Conv2d(in=8, out=64, k=3, bias=True)`: $(3 \times 3 \times 8 + 1) \times 64 = 4'672 \text{ parameters}$
:::

:::{prf:definition} Separable Convolution: Nr of Params

Reduces parameters by splitting kernels into $K \times 1$ and $1 \times K$

A $7 \times 7$ standard filter has 49 weights; a separable one has $7+7=14$
:::

:::{prf:definition} Depthwise Separable Convolution: Nr of Params

1. Convolve each channel separately with a $n \times n$ filter to produce 1 channel output.
2. Then use a $1 \times 1$ filter (called pointwise filter) to mix features across channels.

This drastically reduces computation with minimal performance loss.
:::

In [20]:
class depthwise_separable_conv(nn.Module):
  def __init__(self, in_ch: int, out_ch: int,
      kernel_size = 3,
      padding = 1, bias=False):
    super(depthwise_separable_conv, self).__init__()
    self.depthwise = nn.Conv2d(in_ch, in_ch,
        kernel_size, padding=padding, groups=in_ch,
        bias=bias)
    self.pointwise = nn.Conv2d(in_ch, out_ch,
        kernel_size=1, bias=bias)
  def forward(self, x):
    return self.pointwise(self.depthwise(x))

:::{prf:definition} Bottleneck Layer: Nr of Params

1. Use $1 \times 1$ convolutions to reduce number of channels
2. Do expensive operations (i.e. use $3 \times 3$ convolution) on less channels
3. Use $1 \times 1$ convolutions to increase the number channels

For example: No Bottleneck layers: Conv layer (blue) has $(3*3*256 +1) * 256 = 590'080$ parameters
![Without Bottleneck Layers](Bottleneck-Layers-No.png)

With Bottleneck layers (red): Conv layer (blue) + Bottleneck layers have $(1*1*64 + 1) * 256 + (3*3*64 + 1) * 64 + (1*1*256 + 1) * 64 = 166'400$ parameters
![With Bottleneck Layers](Bottleneck-Layers-Yes.png)
:::

In [21]:
def bottleneck(self, in_ch: int, out_ch: int,
    stride, padding):
    btl_ch = out_ch // 4 # bottleneck channels
    return nn.Sequential(
        nn.BatchNorm2d(in_ch),
        nn.ReLU(True),
        # conv 1x1:
        nn.Conv2d(in_ch, btl_ch, kernel_size=1,
          stride=stride, padding=0),
        # conv 3x3:
        nn.BatchNorm2d(btl_ch),
        nn.ReLU(True),
        nn.Conv2d(btl_ch, btl_ch, kernel_size=3,
          stride=1, padding=padding),
        # conv 1x1:
        nn.BatchNorm2d(btl_ch),
        nn.ReLU(True),
        nn.Conv2d(btl_ch, out_ch, kernel_size=1,
          stride=1, padding=0))

# Auto-Encoder (AE)

The Auto-Encoder is a neural network designed to learn efficient data codings in an unsupervised manner. It forces the network to learn the most significant features of the data by compressing it into a lower-dimensional space.

**Encoder:** Compresses the input $x$ into a small latent vector $z$ using convolutional layers and downsampling (e.g., strided convolutions or pooling).

**Decoder:** Reconstructs the image $\hat{x}$ from the latent vector $z$ using upsampling techniques like **Transposed Convolutions** or **Upsampling** layers.

**Objective:** Minimize the **Reconstruction Loss** (typically Mean Squared Error) between the input image and the reconstructed output.

**Use Case:** Dimensionality reduction, feature extraction, and denoising.

# Recurrent Architectures

Recurrent Architectures such as RNNs, LSTMs and GRUs process sequences step-by-step using a hidden state $H$.

**Vanishing Gradients:** Long sequences involve multiplying weights $W$ many times ($W^n$). If $W < 1$, gradients vanish; if $W > 1$, they explode. _LSTMs_ (Long-Short-Term-Memory) and _GRUs_ (Gated Recurrent Unit) use "gates" to mitigate this and preserve long-term dependencies. LSTMs face several technical challenges, particularly when dealing with long sequences

**Sequential Bottleneck:** Because these architectures process data token-by-token, they are slower and more computationally expensive compared to parallel architectures like Transformers.

**Memory Loss:** On very long sequences, it remains difficult to maintain "long-term dependencies," meaning the model may lose critical information from the start of the sequence by the time it reaches the end.

**Fixed-Vector Bottleneck:** In sequence-to-sequence tasks, forcing an entire input into a single fixed-length hidden state creates a bottleneck that limits performance on complex data.

## RNN: Recurrent Neural Network

$$
h_t = \tanh(x_t W_{ih}^T + b_{ih} + h_{t-1} W_{hh}^T + b_{hh})
$$

where $h_t$ is the hidden state at time $t$, $x_t$ is the input at time $t$, and $h_{(t-1)}$ is the hidden state of the previous layer at time $t-1$ or the initial hidden state at time $0$.

![RNN Cell](RNN-Cell.png)

**Concept:** Designed for processing sequential data (time series, text, audio) where the order matters and input length varies.

**Mechanism:** Unlike feed-forward networks, RNNs have loops. They maintain a **hidden state** ($h$) which acts as a short-term memory. At each time step $t$, the network takes the current input $x_t$ and the previous hidden state $h_{t-1}$ to calculate the output and the new hidden state.

**Key Issue (Vanishing Gradients):** During backpropagation through time (BPTT), gradients are multiplied repeatedly by the weight matrix ($W^n$). For long sequences, if weights are small, gradients vanish to zero (network stops learning); if large, they explode. This makes standard RNNs bad at learning long-term dependencies.

## LSTM: Long Short-Term Memory

$$
\begin{align}
i_t &= \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}) \\
f_t &= \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}) \\
g_t &= \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}) \\
o_t &= \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}) \\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t \\
h_t &= o_t \odot \tanh(c_t)
\end{align}
$$

where $h_t$ is the hidden state at time $t$, $c_t$ is the cell state at time $t$, $x_t$ is the input at time $t$, $h_{t-1}$ is the hidden state of the layer at time $t-1$ or the initial hidden state at time $0$, and $i_t$, $f_t$, $g_t$, $o_t$ are the input, forget, cell, and output gates, respectively. $\sigma$ is the sigmoid function, and $\odot$ is the Hadamard product (element-wise multiplication).

![LSTM Cell](LSTM-Cell.png)

**Solution:** Designed specifically to fix the vanishing gradient problem and capture long-term dependencies.

**Architecture:** Introduces a **Cell State** ($C$) alongside the Hidden State. The Cell State acts as a "highway" for information to flow unchanged if needed.

**Gates:** Uses sigmoid activation "gates" to control information flow:

1.  **Forget Gate:** Decides what to throw away from the cell state.
2.  **Input Gate:** Decides what new information to store in the cell state.
3.  **Output Gate:** Decides what to output based on the cell state and input.

## GRU: Gated Recurrent Unit

$$
\begin{align}
r_t &= \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{(t-1)} + b_{hr}) \\
z_t &= \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{(t-1)} + b_{hz}) \\
n_t &= \tanh(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{(t-1)} + b_{hn})) \\
h_t &= (1 - z_t) \odot n_t + z_t \odot h_{(t-1)}
\end{align}
$$

where $h_t$ is the hidden state at time $t$, $x_t$ is the input at time $t$, $h_{(t-1)}$ is the hidden state of the layer at time $t-1$ or the initial hidden state at time $0$, and $r_t$, $z_t$, $n_t$ are the reset, update, and new gates, respectively. $\sigma$ is the sigmoid function, and $\odot$ is the Hadamard product.

![GRU Cell](GRU-Cell.png)

**Concept:** A simplified, more efficient variation of the LSTM.
**Architecture:** Merges the Cell State and Hidden State into a single state. Combines the Forget and Input gates into a single **Update Gate**. Adds a **Reset Gate** to decide how much past information to forget.

**Comparison:** GRUs have fewer parameters than LSTMs, making them faster to train and often performing just as well on smaller datasets, though LSTMs may be more powerful for very complex tasks.

# Transformers

## The Attention Mechanism

Attention allows a model to "look back" at all positions of an input sequence simultaneously, solving the bottleneck of single-vector encodings.

**The Trinity:** $Query$ (what I am looking for), $Key$ (what I have), and $Value$ (the information content)
$$Q = XW^Q, K = XW^K, V = XW^V$$

**Attention Score:** The scaled dot-product between $Q$ and $K$ ($d_k := \text{Dimension of } K$), the $\sqrt{d_k}$ factor keeps variance at 1, ensuring **stable gradients** regardless of the input dimension $d_k$:

$$E = \frac{QK^T}{\sqrt{d_k}}$$

**Attention:**

$$Attention(Q, K, V) = \text{softmax}\left(E\right)V$$

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^d e^{z_j}} \quad \text{softmax: } \mathbb{R}^d \to (0,1)^d$$

:::{prf:definition} Output Size
The Output Size (or Embedding Dimension) is the size of the vector representing each token as it enters and leaves the layer ($d_k := \text{Dimension of } K$):

$$
d_{model} = n_{heads} \times d_k
$$

**Scenario:** Input of 10 tokens, length 512, 4 layers, 8 heads.

**Result:** The output dimension remains **10 tokens $\times$ 512 length**.

(Self-) Attention preserves the sequence length and embedding dimension ($d_{model}$) regardless of the number of heads or layers (because we adjust $d_k$ based on number of heads: $d_k = \frac{d_{model}}{n_{heads}}$). This allows us to stack as many layers as we want without the vector shrinking or growing.
:::

## Transformer Architecture and its Goals

Minimize computational complexity per layer

Minimize path length between pair of words to facilitate learning of long-range dependencies

Maximise the amount of computation that can be parallelized

![Transformer Architecture](Transformer-Architecture.png)

**Self-Attention:** $Q$, $K$, and $V$ all stem from the same input sequence via different learnable linear transforms.

**Multi-Head Attention:** Uses multiple sets of $(Q, K, V)$ to focus on different aspects of the sequence in parallel.

**Positional Encoding:** Since self-attention is permutation-invariant (order doesn't matter), unique sine/cosine vectors are added to input embeddings to inject sequence order.

**Decoder Training:** In order that the decoder cannot cheat (in parallel  training) and look at future tokens, we have to mask  them out

**Vision Transformer (ViT):** Images are split into **patches** (e.g., $16 \times 16$), which are flattened and treated as tokens in a sequence for the transformer encoder.

![Vision Transformer Architecture](Vision-Transformer-Architecture.png)

# Reinforcement Learning (RL)

![Taxonomy of Reinforcement Learning Algorithms](RL-Taxonomy-Algos.png)

# Reinforcement Learning Notation

$A_t$ action at time $t$

$S_t$ state at time $t$, typically due, stochastically, to $S_{t-1}$ and $A_{t-1}$

$R_t$ reward at time $t$, typically due, stochastically, to $S_{t-1}$ and $A_{t-1}$

$\pi$ policy (decision-making rule)

$\pi(s)$ action taken in state $s$ under deterministic policy $\pi$

$\pi(a \mid s)$ probability of taking action $a$ in state $s$ under stochastic policy $\pi$

$G_t$ return following time $t$

$v_\pi(s)$ value of state $s$ under policy $\pi$ (expected return)

$v_*(s)$ value of state $s$ under the optimal policy

$q_\pi(s, a)$ value of taking action $a$ in state $s$ under policy $\pi$

$q_*(s, a)$ value of taking action $a$ in state $s$ under the optimal policy

**Reinforcement learning methods specify how the agent's policy is  changed as a result of its experience.**

:::{prf:definition} V: State-Value Function
Estimates the expected reward of a given state:

$$v_\pi(s) := \mathbb{E}_\pi [ G_t \mid S_t = s ] , \quad \text{for all } s \in \mathcal{S}$$

Or calculate iteratively: $V(S_t) = V(S_t) + \alpha[G_t - V(S_t)]$
:::

:::{prf:definition} Q: Action-Value Function
Estimates the expected reward of an action for a given state:

$$q_\tau(s, a) := \mathbb{E}_\tau [ G_t \mid S_t = s, A_t = a ]$$

The best estimation is taking the average:

$$Q_n = \frac{R_1 + R_2 + \cdots + R_{n-1}}{n-1}$$

Or calculate iteratively: $Q_{n+1} = Q_n + \frac{1}{n} (R_n - Q_n)$
:::

# Markov Decision Processes (MDP)

**Goal:** Maximize the **Return** ($G_t$), the cumulative (often discounted) reward:
    $$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots$$

:::{prf:definition} Dynamics of an MDP
$$
p(s', r \mid s, a) \doteq \Pr\{ S_t = s', R_t = r \mid S_{t-1} = s, A_{t-1} = a \}
$$

**Markov Property:** The future is independent of the past given the present (The future depends only on the current state, not the history).
:::

:::{prf:definition} Bellman Equation
$$
v_\pi(s) = \sum_a \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\,[r + \gamma v_\pi(s')], \quad \text{for all } s \in \mathcal{S}
$$

$\pi(a \mid s)$ is the policy, $p(s', r \mid s, a)$ is the MDP, $r$ is the reward of the current state, $\gamma v_\pi(s')$ is the discounted future reward
:::

# RL Key Algorithms

:::{prf:definition} Epsilon Greedy
To balance exploitation vs. exploration, the agent chooses actions based on a $\epsilon$-greedy policy:

**Exploitation:** With probability $1 - \epsilon$, take greedy action (max $Q_t(a)$)

**Exploration:** With probability $\epsilon$, take any valid action with uniform probability
:::

:::{prf:definition} Monte Carlo Prediction for estimating $v_{\pi}$
**Input:**
- a policy $\pi$

**Initialize:**
- $V(s) \in \mathbb{R}$ `#arbitrarily`
- $Returns(s) \gets$ an empty list, for all $s \in \mathcal{S}$

**Loop forever:**
- $\pi$: $S_0, A_0, R_0, S_1, \dots, R_T$
- $G \gets 0$ `#Cumulative reward or return`
- Loop for each time step $t = T-1, T-2, \dots, 0$:
  - $G \gets \gamma G + R_{t+1}$
  - Unless $S_t$ appears in $S_{t - 1}, \dots, S_{0}$: `#first visit only`
    - Append $G$ to $Returns(S_t)$
    - $V(S_t) \gets \text{average}(\text{Returns}(S_t))$

`HINT: Its easier to calculate the total reward backwards. The last state does not yield any reward so its 0.`
:::

:::{prf:definition} TD(0) for estimating $v_{\pi}$
**Temporal Difference Learning**

**Input:**
- the policy $\pi$ to be evaluated
- step size $\alpha \in (0, 1]$

**Initialize:**
- $V(S)$ arbitrarily (except $V(terminal = 0)$)

**Loop for each episode:**
- Initialize $S$
- Loop for each step of episode:
  - $A \gets$ action given by $\pi$ for $S$
  - Take action $A$ and observe state $S'$ and reward $R$
  - $V(S) \gets V(S) + \alpha [R + \gamma V(S') - V(S)$
  - $S \gets S'$
- until $S$ is terminal
:::

:::{prf:definition} On-policy first-visit MC control, estimates $\pi \approx \pi_*$
**Input:**
- (small) $\epsilon > 0$

**Initialize:**
- $\pi \leftarrow$ an arbitrary $\epsilon$-soft policy
- $Q(s, a) \in \mathbb{R}$ (arbitrarily, for example $= 0$)
- $\text{Returns}(s, a) \leftarrow$ empty list

**Loop forever** (for each episode):
- Generate an episode following $\pi$: $S_0, A_0, R_0, S_1, \ldots, R_T$
- $G \leftarrow 0$
- Loop for each step of the episode, $t = T-1, T-2, \ldots, 0$:
  - $G \leftarrow \gamma G + R_{t+1}$
  - Unless $(S_t, A_t)$ appears in $(S_{t-1}, A_{t-1}), \ldots, (S_0, A_0)$:
    - Append $G$ to $\text{Returns}(S_t, A_t)$
    - $Q(S_t, A_t) \leftarrow \text{average}(\text{Returns}(S_t, A_t))$
    - $A^* \leftarrow \arg\max_a Q(S_t, a)$, ties broken arbitrarily
    - For all $a \in \mathcal{A}(S_t)$:\
      $\pi(a \mid S_t) \leftarrow \begin{cases}
      1 - \epsilon + \epsilon / |\mathcal{A}(S_t)| & \text{if } a = A^* \\
      \epsilon / |\mathcal{A}(S_t)| & \text{otherwise}
      \end{cases}$
:::

:::{prf:definition} SARSA for estimating $Q \approx q_*$
**Input:**
- step size $\alpha \in (0, 1]$
- small $\epsilon > 0$

**Initialize:**
- $Q(s, a)$ for all $s \in \mathcal{S}^+, a \in \mathcal{A}$ arbitrarily (except $Q(\text{terminal}, \cdot) = 0$)

**Loop for each episode:**
- Initialize $S$
- Choose $A$ from $S$ using a policy derived from $Q$ (e.g., $\epsilon$-greedy)
- Loop for each step of the episode:
  - Take action $A$, observe $R, S'$
  - Choose $A'$ from $S'$ using a policy derived from $Q$ (e.g., $\epsilon$-greedy)
  - $Q(S, A) \leftarrow Q(S, A) + \alpha[R + \gamma Q(S', A') - Q(S, A)]$
  - $S \leftarrow S'$; $A \leftarrow A'$
- until $S$ is terminal
:::

:::{prf:definition} Q-Learning for estimating $Q \approx q_*$
**Input:**
- step size $\alpha \in (0, 1]$
- small $\epsilon > 0$

**Initialize:**
- $Q(s, a)$ for all $s \in \mathcal{S}^+, a \in \mathcal{A}$ arbitrarily (except $Q(\text{terminal}, \cdot) = 0$)

**Loop for each episode:**
- Initialize $S$
- Loop for each step of the episode:
  - Choose $A$ from $S$ using a policy derived from $Q$ (e.g., $\epsilon$-greedy)
  - Take action $A$, observe $R, S'$
  - $Q(S, A) \leftarrow Q(S, A) + \alpha[R + \gamma \max_a Q(S', a) - Q(S, A)]$
  - $S \leftarrow S'$
- until $S$ is terminal
:::

:::{prf:definition} Gradient Monte Carlo Prediction
**Input:**
- a policy $\pi$
- a differentiable value-function $\hat{v} : \mathcal{S} \times \mathbb{R}^d \to \mathbb{R}$ with parameters $\mathbf{w}$
- a step size parameter $\alpha > 0$

**Initialize:**
- $\mathbf{w}$ arbitrarily

**Loop forever:**
- Generate episode following $\pi$: $S_0, A_0, R_0, S_1, \ldots, R_T$
- $G \leftarrow 0$
- Loop for each step of the episode, $t = T-1, T-2, \ldots, 0$:
  - $G \leftarrow \gamma G + R_{t+1}$
  - $\mathbf{w} \leftarrow \mathbf{w} + \alpha [G_t - \hat{v}(S_t, \mathbf{w})] \nabla \hat{v}(S_t, \mathbf{w})$
:::

:::{prf:definition} Deep Q-Learning with experience replay
**Initialize:**
- the replay memory $\mathcal{D}$ to capacity $N$
- action-value function $Q$ with random weights $\mathbf{w}$
- target action-value function $\hat{Q}$ with weights $\mathbf{w}^- = \mathbf{w}$

**Loop for each episode:**
- Initialize $S_1$
- For every step $t = 1, T$ in the episode:
  - Choose $A_t$ as a function of $Q(S_t, \cdot, \mathbf{w})$ (e.g., $\epsilon$-greedy)
  - Take action $A_t$, observe $R_t, S_{t+1}$
  - Store transition $(S_t, A_t, R_t, S_{t+1})$ in $\mathcal{D}$
  - Sample a random minibatch of transitions $(S_j, A_j, R_j, S_{j+1})$ from $\mathcal{D}$
  - Set target:
    $$y_j = \begin{cases}
    R_j & \text{if } S_{j+1} \text{ is terminal} \\
    R_j + \gamma \max_{A'} \hat{Q}(S_{j+1}, A', \mathbf{w}^-) & \text{otherwise}
    \end{cases}$$
  - Perform a gradient step on $(y_j - Q(S_j, A_j, \mathbf{w}))^2$ with respect to $\mathbf{w}$
  - Every $C$ steps reset $\hat{Q} = Q$
:::

:::{prf:definition} Hill climbing to find best policy function
**Input:**
- a policy parameterization $\pi(a \mid s, \boldsymbol{\theta})$

**Initialize:**
- policy parameters $\boldsymbol{\theta}$
- Generate an episode following $\pi$ to obtain $G$
- $\boldsymbol{\theta}_{\text{best}} \leftarrow \boldsymbol{\theta}$, $G_{\text{best}} \leftarrow G$

**Repeat:**
- Add random noise to $\boldsymbol{\theta}_{\text{best}}$ to get $\boldsymbol{\theta}_{\text{new}}$
- Generate an episode following $\pi$ to obtain $G_{\text{new}}$
- If $G_{\text{new}} > G_{\text{best}}$:
  - $\boldsymbol{\theta}_{\text{best}} \leftarrow \boldsymbol{\theta}_{\text{new}}$, $G_{\text{best}} \leftarrow G_{\text{new}}$
:::

:::{prf:definition} One step Actor-Critic (episodic) for estimating $\pi \approx \pi_*$
**Input:**
- a differentiable policy parameterization $\pi(a \mid s, \boldsymbol{\theta})$
- a differentiable state-value function parameterization $\hat{v}(s, \mathbf{w})$
- step sizes $\alpha_{\boldsymbol{\theta}} > 0, \alpha_{\mathbf{w}} > 0$

**Initialize:**
- policy parameters $\boldsymbol{\theta}$ and state-value weights $\mathbf{w}$

**Loop for each episode:**
- Initialize $S$, first state of episode
- $I \leftarrow 1$
- For each time step of the episode:
  - Choose $A \sim \pi(\cdot \mid S, \boldsymbol{\theta})$
  - Take action $A$, observe $S', R$
  - $\delta \leftarrow R + \gamma \hat{v}(S', \mathbf{w}) - \hat{v}(S, \mathbf{w})$
  - $\mathbf{w} \leftarrow \mathbf{w} + \alpha_{\mathbf{w}} \delta \nabla \hat{v}(S, \mathbf{w})$
  - $\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} + \alpha_{\boldsymbol{\theta}} I \delta \nabla \ln \pi(A \mid S, \boldsymbol{\theta})$
  - $I \leftarrow \gamma I$
  - $S \leftarrow S'$
:::

:::{prf:definition} PPO (Proximal Policy Optimization)
Uses a **clipped surrogate loss** to prevent the new policy from diverging too far from the old one, ensuring stable training.
:::

# GNN: Graph Neural Networks

Deep learning architectures designed to process data represented as graphs (nodes and edges), such as molecules, social networks, or road maps. Unlike standard CNNs which operate on fixed grids, GNNs must handle irregular structures and varying neighborhood sizes.

**Key Property:** GNNs must be **Permutation Invariant**. The output must remain the same regardless of the order in which nodes are indexed in the input matrices.

**Approaches:**

_Spectral Methods:_ Define convolutions in the frequency domain using the Graph Laplacian (e.g., GCN)

_Spatial / Message Passing:_ Nodes aggregate information ("messages") from their direct neighbors to update their own features (e.g., GraphSAGE).

## Adjacency Matrix

The Adjacency Matrix ($A$) is a square $N \times N$ matrix (where $N$ is the number of nodes) that mathematically represents the graph's connectivity. **Calculation Steps:**

1.  Initialize a matrix of zeros with dimensions $N \times N$.
2.  For every connection (edge) between Node $i$ and Node $j$, set the value at row $i$, column $j$ ($A_{ij}$) to **1**.
3.  If the graph is **undirected**, ensure symmetry: if $A_{ij} = 1$, then $A_{ji} = 1$.
4.  *Self-Loops:* For many GNNs (like GCN), we add self-loops so a node considers its own features during updates. This means setting the diagonal elements to 1 ($A_{ii} = 1$).

**Example:** Given a graph with 3 nodes where Node 1 is connected to Node 2, and Node 2 is connected to Node 3 (Note: With self-loops added, the diagonal would become ones):

$$
\mathbf{A} = \begin{array}{c|ccc}
  & N_1 & N_2 & N_3 \\
\hline
N_1 & 0 & 1 & 0 \\
N_2 & 1 & 0 & 1 \\
N_3 & 0 & 1 & 0 \\
\end{array}
=
\begin{bmatrix}
0 & 1 & 0 \\
1 & 0 & 1 \\
0 & 1 & 0
\end{bmatrix}
$$

## GNN for Graph Classification

Graph classification predicts a label for the **entire graph** (e.g., "Is this molecule toxic?") rather than for individual nodes. **Architecture:**

1.  **Input:** Adjacency Matrix $A$ and Node Features $X$.
2.  **GNN Layers (Message Passing):** Several layers of graph convolutions update the embeddings for every node based on their neighbors. This captures local structure.
3.  **Readout (Global Pooling):** A global aggregation step is required to compress all node embeddings into a single **Graph Embedding**. Common methods include **Global Mean Pooling** (average of all nodes) or Global Max Pooling.
4.  **MLP Classifier:** The resulting single graph vector is passed through standard Dense/Linear layers.
5.  **Output:** A Softmax layer produces the final class probability.

$$ \text{Graph} \xrightarrow{GNN} \{h_1, h_2... h_N\} \xrightarrow{Pooling} h_{graph} \xrightarrow{MLP} \text{Class} $$

# Explainable AI Nomenclature

**Global vs. Local:** Global explanations describe a model's behavior across an entire dataset, while local explanations justify a specific prediction for a single instance.

**Intrinsic vs. Post-hoc:** Intrinsic models, also called glassbox or white-box models, are transparent by design (e.g., decision trees), whereas post-hoc methods explain "black-box" models after they have been trained.

**Model-Agnostic vs. Model-Specific:** Model-agnostic techniques can be applied to any architecture because they only use inputs and outputs, while model-specific methods rely on internal components like convolutional feature maps.

**Surrogate Models:** These are simple, interpretable models trained to approximate a complex model's behavior within a localized parameter space.

**Attributions and Saliency Maps:** These terms refer to visualizations, often heatmaps, that highlight which parts of an input most influenced a specific model decision.

**Counterfactuals:** These provide "what-if" scenarios by identifying the smallest change needed in an input to flip the model's output.

# XAI Challenges

The primary challenges of XAI stem from the fundamental tension between model complexity and transparency: as models become more accurate, they generally become more opaque.

**The Missingness Problem:** It is difficult to define "nothingness" for a neural network without biasing the output. For example, using a black baseline (zeros) may be highly meaningful in certain data types, like sketches.

**The Correlation Problem:** Many methods assume features are independent, which can lead to non-physical explanations that do not respect the data's true distribution.

**Saturation and Vanishing Gradients:** In well-trained models, the output can be so certain that small changes to individual pixels result in near-zero gradients, making sensitivity-based explanations noisy.

**Computational Expense:** Techniques like occlusion require thousands of forward passes, while calculating exact Shapley values is mathematically intractable due to the $2^n$ possible feature combinations.

**Resolution vs. Semantics:** Model-specific methods like CAM and Grad-CAM face a trade-off where deeper layers provide high-level semantic meaning but suffer from low spatial resolution because the feature maps have shrunk.

**Sensitivity to Hyperparameters:** Explanations can vary drastically based on user-selected values, such as the kernel width in LIME or the chosen baseline in Integrated Gradients.

# Glassbox Models

Glassbox or "white box" models are models that are intrinsically interpretable, not requiring additional techniques to understand their decisions. Common examples include linear models, where weights directly represent feature importance, and decision trees. Although these models often sacrifice accuracy for transparency, they serve as critical components in more advanced XAI methods like LIME and CAM.

# XAI Plot Based Methods

Individual Conditional Expectation (ICE or ICPs) is a local explanation method that visualizes how a model's prediction changes for a specific instance as one feature is varied,. While Partial Dependence Plots (PDPs) show an overall average effect across a dataset, ICE plots a separate curve for each instance to reveal variations that might be hidden by aggregation,.

The procedure involves selecting a feature and sweeping its unique values for a single row while keeping all other feature values constant,. This allows researchers to identify heterogeneous effects, such as subsets of data that react differently to feature changes compared to the global average.

![PDP](Partial-Dependency-Plot.png)
![ICE Plot](Individual-Conditional-Expectation-Plot.png)

# Saliency Mapping

Saliency mapping (also known as heatmap or attribution mapping) identifies which parts of an input were most influential in a model's specific prediction.

## Occlusion and Adaptive Occlusion

Occlusion is a simple, **model-agnostic** method that identifies critical regions by "blacking out" portions of an input and measuring the change in the model's output (Adaptive Occlusion attempts to minimize the occluded area while maintaining the same prediction output w/o occlusion). It produces a saliency map where regions that cause the largest prediction swings are highlighted as important. Its main drawbacks include high computational costs and extreme sensitivity to hyperparameters like kernel size and the chosen baseline value (e.g., black vs. noise).

## GAP: Global Average Pooling

Global Average Pooling (GAP) is a dimensionality reduction technique that computes the average value across all spatial dimensions of a feature map, reducing a 3D tensor to a 1D vector. It replaces fully connected layers at the end of convolutional neural networks.

**Mathematical Definition:** For a feature map with shape $(C, H, W)$ where $C$ is the number of channels, $H$ is height, and $W$ is width:

$$z_c = \frac{1}{H \times W} \sum_{i=1}^{H} \sum_{j=1}^{W} f_c[i, j]$$

where $z_c$ is the output for channel $c$, and $f_c[i,j]$ is the value at position $(i,j)$ in channel $c$.

**Output shape:** $(C,)$ — a single scalar value per channel.

## CAM: Class Activation Mapping

CAM is a **model-specific**: It requires a specific architecture (Global Average Pooling (GAP) followed by a linear layer). The convolutional feature maps are processed through Global Average Pooling (GAP) to create a linear model just before the final prediction. By multiplying these feature maps by their corresponding class weights and summing them (often using a ReLU), the model generates a heat map of the discriminant regions. While semantic quality improves in deeper layers, the resolution of these explanations tends to decrease as the maps get smaller.

:::{prf:example} Calculation Example
Weights $w=[2.0, -1.0, 0.5]$ for feature maps with values $[0.5, 0.0, 2.0]$ at pixel $(i,j)$

$$CAM = \text{ReLU}((2.0 \times 0.5) + (-1.0 \times 0.0) + (0.5 \times 2.0)) = 2.0$$
:::

![Class Activation Mapping (CAM)](Class-Activation-Mapping.png)

## Grad-CAM

Is more flexible than standard CAM because it uses **gradients** to weigh the importance of feature maps rather than requiring a specific architecture. Importance is assessed by "wiggling" individual feature maps and measuring the sensitivity of the output.

![Grad-CAM](Grad-CAM.png)

Frees up architecture constraints; Don’t have to compromise accuracy as final conv layer can go into any function, not just a SoftMax; Final feature map resolution dictates the graduality of the explanation; Tension between high level semantics and explanatory resolution; Cannot compare the intensity of final heatmaps between different instances; Still requires CNN.

## LIME: Local Interpretable Model-Agnostic Explanations

LIME  is a **model-agnostic** method that provides "local" explanations for individual instances, working across tabular, text, and image data. It operates by randomly sampling data around a specific prediction and weighting those samples using a proximity measure - typically an exponential kernel $\exp(-D(x,z)^2 / \sigma^2)$ - so that points closer to the original input are more important. A simple, interpretable **surrogate model** (such as a linear model) is then trained to mimic the black-box model's behavior in that local neighborhood.

:::{figure} LIME-Idea.png
Basic Idea of LIME
:::

:::{figure} LIME-Generalized.png
**Left:** Blackbox-Model, **Right:** Surrogate Model, **Bottom:** Enforces locality for predictions: If the distance is large, it should be less relevant if the prediction is different.
:::

# Local vs. Global Explanations

**Local:** Explains a specific instance (e.g., "Why was *this* loan denied?").

**Global:** Summarizes feature importance across the entire dataset.

# SHAP: Shapley Values

Based on game theory: how to fairly distribute the "payout" (prediction) among "players" (features).

**Axiom of Completeness:** The sum of all feature attributions must equal the model's prediction minus the baseline.

**Integrated Gradients:** A path-integral method that integrates gradients along a line from a baseline (e.g., black image) to the target image to solve the **saturation problem** (where gradients go to zero in trained models).

# Generative AI

## Explicit vs. Implicit Density

**Explicit:** Models the probability $P(x)$ directly (e.g., PixelRNN, VAE).

**Implicit:** Learns to sample from the distribution without explicitly calculating $P(x)$ (e.g., GANs).

## VAE: Variational Auto-Encoder

The VAE is a **generative model** that extends the AE structure to estimate the probability density of the data. Instead of mapping an input to a fixed point in the latent space, it maps it to a probability distribution. Optimizes the Evidence Lower Bound (ELBO). The Encoder approximates the posterior $q(z|x)$, and the KL divergence forces it close to a Gaussian prior.

**Latent Space:** The encoder outputs parameters for a distribution (Mean $\mu$ and Variance $\sigma$) rather than a single vector. The system then samples $z$ from this distribution to feed the decoder.

**Smoothness:** The latent space is continuous, allowing for valid interpolation between points (e.g., morphing one image into another), unlike standard AEs.

**Trade-off:** VAEs are easy to train and fast to sample from, but often produce blurrier images compared to GANs,.

**Loss Function (ELBO):**
The VAE optimizes the **Evidence Lower Bound (ELBO)**, which consists of two competing terms:

1. **Reconstruction Loss:** Ensures the output resembles the input (Likelihood).
2. **KL Divergence:** A regularization term that forces the learned latent distribution $q(z|x)$ to approximate a standard Gaussian prior $p(z)$ (typically $\mathcal{N}(0,1)$).

$$ \mathcal{L} = \mathbb{E}_{q}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

**Equation Insight:** If the encoder models a Gaussian distribution, the KL term has a simple analytic expression involving $\mu$ and $\sigma$:

$$D_{KL}(q(z|x) || p(z)) = \frac{1}{2} \sum_{j=1}^{M} \left( 1 + \ln(\sigma_j^2x) - \mu_j^2x - \sigma_j^2x \right)$$

*   **Inputs:** The encoder network directly outputs the mean ($\mu$) and variance ($\sigma^2$) for each latent dimension $j$.
*   **Result:** This formula penalizes the network if:
    *   $\mu$ diverges from **0** (the $-\mu^2$ term).
    *   $\sigma^2$ diverges from **1** (the $1 + \ln(\sigma^2) - \sigma^2$ terms).

This insight refers to a computational shortcut that makes training Variational Autoencoders (VAEs) efficient.

**Reparameterization Trick:**

_The Problem:_ In a standard VAE, the encoder outputs the parameters of a distribution (mean $\mu$ and variance $\sigma$). The network must then **sample** a latent vector $z$ from this distribution to feed the decoder. This sampling operation is stochastic (random). Standard backpropagation cannot compute gradients through a random node, meaning the encoder's weights cannot be updated to minimize the error.

_The Solution:_ The reparameterization trick solves this by expressing the random variable $z$ as a deterministic function of the model parameters and an independent source of noise. Instead of sampling $z$ directly from $\mathcal{N}(\mu, \sigma^2)$, the model:

1.  Samples a noise vector $\epsilon$ from a fixed standard normal distribution: $\epsilon \sim \mathcal{N}(0, 1)$
2.  Calculates $z$ using the deterministic formula:
    $$ z = \mu + \sigma \cdot \epsilon $$

The randomness is now contained in $\epsilon$, which is treated as a constant input during backpropagation. The latent vector $z$ is now a differentiable function of $\mu$ and $\sigma$, allowing gradients to flow from the decoder back into the encoder.

## GAN: Generative Adversarial Network

**The MinMax Game:** A **Generator** creates fake images from noise, and a **Discriminator** tries to distinguish them from real data.

**Minmax Loss:**
$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

**WGAN:** Uses the **Earth Mover (Wasserstein) distance**, to provide a stable gradient even if the generator and data distributions do not overlap.

**CycleGAN:** Learns style transfer between two domains without paired data (e.g., horses to zebras) using a **Cycle Consistency Loss** ($F(G(x)) \approx x$) to allow style transfer without paired training data.

# SSL: Self-Supervised Learning

## Pretext Tasks
The model solves "fake" tasks to learn a **backbone** representation.

Examples: Predicting image rotation, solving jigsaw puzzles, or colorization.

Success is measured by performance on **downstream tasks** (e.g., classification) using only a small amount of labeled data.

## SimCLR: Contrastive Learning

**Method:** Create two augmented versions of the same image (positive pair) and maximize their similarity while minimizing similarity with all other images in the batch (negative pairs).

**Order of Operations:** Data Augmentation $\to$ Encoder $\to$ Projection Head $\to$ Contrastive Loss.

**Projection Head:** A small MLP that "absorbs" the contrastive loss, allowing the encoder to maintain general features.

## BYOL: Bootstrap Your Own Latent

Learns representations **without negative examples**.

Uses two asymmetric networks (**Online** and **Target**). The Target weights are an **Exponential Moving Average (EMA)** of the Online weights, preventing the model from collapsing to a trivial constant output and negative pairs.